# 采集真实 120 波形 → 存 npy → 打印检测器间隔(定位精度问题的关键)

运行后**务必把 `/home/xilinx/jupyter_notebooks/cap_120_t.npy` 发给我**。音量:swing 控制在 ~3000~6000。

In [ ]:
import time, struct, numpy as np
from pynq import Overlay, allocate
import smbus2
from smbus2 import i2c_msg
GAIN=0x20; LEN=512; Fs=46300.0; BLOCKS=1500   # ~1500*256=384k样本 ≈8.3s
ol=Overlay('design_1.bit')
seg,bpm,dma,adau=ol.segment_stream_0,ol.bpm_detect_0,ol.axi_dma_0,ol.adau1761_0
bus=smbus2.SMBus(0); CAD=0x3B
def wr16(r,v):
    bus.i2c_rdwr(i2c_msg.write(CAD,[(r>>8)&0xFF,r&0xFF,v])); time.sleep(0.002)
wr16(0x4000,0x00); wr16(0x4000,0x03); wr16(0x40F9,0x7F); wr16(0x40FA,0x01)
wr16(0x4015,0x00); wr16(0x4016,0x00); wr16(0x4017,0x00)
for r,v in [(0x400A,0x11),(0x400B,0x20),(0x400C,0x11),(0x400D,0x20),
            (0x400E,0xFF),(0x400F,0xFF),(0x4010,0x01),(0x4026,0x33)]: wr16(r,v)
print('[1] codec ok')

In [ ]:
buf=allocate(shape=(LEN,),dtype=np.uint32)
dma.recvchannel.start()
ch=np.empty((BLOCKS*LEN//2,),dtype=np.uint32)
print('>> PLAY 120 metronome, recording %d blocks...'%BLOCKS)
for i in range(BLOCKS):
    try: dma.recvchannel.transfer(buf)
    except Exception: pass
    buf.flush(); seg.mmio.write(0x00,LEN); time.sleep(0.005)
    buf.invalidate()
    blk=np.asarray(buf).copy()
    ch[i*LEN//2:(i+1)*LEN//2]=blk[::2]
print('cap %d samples  swing=%d'%(len(ch),int(ch.max())-int(ch.min())))
np.save('/home/xilinx/jupyter_notebooks/cap_120_t.npy', ch)
print('SAVED cap_120_t.npy  -> 请下载发我')

In [ ]:
# ---- 打印 3 个检测器在真实信号上的间隔(诊断根因) ----
import heapq, numpy as np
ch=np.load('/home/xilinx/jupyter_notebooks/cap_120_t.npy').astype(np.int64)
FS=46300.0; BC=int(FS*60); EW=1024; HB=64; BW=640; REF=BC//240
LO=BC//180; HI=BC//60; ROM=[]
raw=ch.astype(np.int64); sig=np.where(raw<0x800000,raw,raw-0x1000000)
def det(s,ADAPT,BSHIFT):
    env=[0]*EW;ei=0;prev=0;base=0;es=0;lp=0;cnt=0
    hist=[0]*HB;last=[0]*HB;bd2=bd1=0;td1=0;ivs=[]
    for n in range(len(s)):
        x=s[n]; dx=x-prev; prev=x; e=dx*dx
        es-=env[ei]; env[ei]=e; es+=e; ei=(ei+1)&(EW-1)
        base+=(es-base)>>BSHIFT; thr=base*ADAPT
        fire=(bd1>td1) and (bd1>=bd2) and (bd1>es) and (cnt-1-lp)>=REF
        nz=cnt-1
        bd2=bd1; bd1=es; td1=max(0,thr)
        if fire:
            iv=nz-lp; lp=nz; ivs.append(iv)
            if LO<=iv<=HI:
                bi=(iv-LO)//BW
                if bi<HB:
                    age=nz-last[bi]; h=age>>17
                    hist[bi]=(0 if h>=32 else hist[bi]>>h)+1
                    last[bi]=nz
        cnt+=1
    return ivs,hist
for name,AD,BS in (("当前IP AD=4",4,17),("弱阈值AD=2",2,17),("快速AD=4 BS=14",4,14)):
    ivs,h=det(sig,AD,BS)
    bb=int(np.argmax(h)) if any(h) else None
    top=heapq.nlargest(4,enumerate(h),key=lambda t:t[1])
    tops=', '.join('%d->%.0f(v%d)'%(i,BC/(LO+i*BW+BW//2),v) for i,v in top if v>0)
    nv=sum(1 for i in ivs if LO<=i<=HI)
    print('
[%s] 触发总=%d 入库=%d 众数BPM=%s'%(name,len(ivs),nv, (BC/(LO+bb*BW+BW//2)) if bb is not None else None))
    print('  topbins:',tops)
    print('  全部间隔->BPM:', ','.join('%d->%d'%(i,round(BC/i)) for i in ivs[:60]))
print('
真120间隔=%.0f样本'%(BC/120.0))